## 3. Transformation

This phase involves cleaning, standardizing, and applying necessary business logic to the extracted raw data.

**Key transformation tasks**

- **Currency conversion:** Merge latest currency exchange rates to calculate local prices.
- **Delivery metrics:** Add columns for late deliveries and latency days.
- **Locality flag:** Determine if customers are local relative to stores.
- **Lookup tables:** Resolve ambiguous columns (e.g. order statuses).
- **Transformed data staging:** Write outputs to `staging_2`.

This notebook implements **currency conversion** on `order_items` using `staging_1` inputs.

### Currency conversion — assumptions

- `exchange_rates.csv` uses **USD as base**: `Rate` is **units of that currency per 1 USD** (USD row has `Rate == 1.0`).
- `order_items.list_price` is treated as **USD**; `discount` is a **fraction** (e.g. `0.2` → 20% off).
- **Line total USD:** `quantity * list_price * (1 - discount)`.
- **Local amount:** `line_total_usd * Rate` for each target currency.
- Staging data has **no per-customer currency**; we add **example columns** (USD, AED, EUR, GBP) for reporting. Adjust `REPORT_CURRENCIES` as needed.

In [1]:
from pathlib import Path

import pandas as pd

# Paths: this notebook lives in Transformation/Code/
TRANSFORMATION_ROOT = Path("..").resolve()
STAGING_1 = TRANSFORMATION_ROOT / "staging_1"
STAGING_2 = TRANSFORMATION_ROOT / "staging_2"
STAGING_2.mkdir(parents=True, exist_ok=True)

PATH_ORDER_ITEMS = STAGING_1 / "order_items.csv"
PATH_ORDERS = STAGING_1 / "orders.csv"
PATH_FX = STAGING_1 / "exchange_rates.csv"

In [2]:
def load_exchange_rates(path: Path) -> pd.DataFrame:
    """Load FX snapshot; drop stray index column if present."""
    fx = pd.read_csv(path)
    # File may start with an unnamed index column (0,1,2,...)
    if fx.columns[0].startswith("Unnamed") or fx.columns[0] == "":
        fx = fx.drop(columns=fx.columns[0])
    elif fx.columns[0] not in ("Currency", "currency") and fx.iloc[:, 0].dtype.kind in "iu":
        first = fx.columns[0]
        if first not in ("Currency", "Rate"):
            fx = fx.drop(columns=first)
    fx = fx.rename(columns={c: c.strip() for c in fx.columns})
    return fx


fx_raw = load_exchange_rates(PATH_FX)
fx = fx_raw[["Currency", "Rate"]].copy()
fx["Currency"] = fx["Currency"].astype(str).str.upper().str.strip()
fx["Rate"] = pd.to_numeric(fx["Rate"], errors="coerce")
fx = fx.drop_duplicates(subset=["Currency"], keep="last")

fx_snapshot_ts = (
    pd.to_datetime(fx_raw["extraction_timestamp"], errors="coerce").max()
    if "extraction_timestamp" in fx_raw.columns
    else pd.NaT
)
print(f"FX rows: {len(fx)} | snapshot (max extraction_timestamp): {fx_snapshot_ts}")
fx.head()

FX rows: 172 | snapshot (max extraction_timestamp): 2026-04-01 21:42:12.252244


,Currency,Rate
0,AED,3.673150
1,AFN,62.999998
2,ALL,82.659229
3,AMD,377.230000
4,ANG,1.790000


In [3]:
order_items = pd.read_csv(PATH_ORDER_ITEMS)
orders = pd.read_csv(PATH_ORDERS)

lines = order_items.merge(orders, on="order_id", how="left", suffixes=("", "_order"))

for col in ("quantity", "list_price", "discount"):
    lines[col] = pd.to_numeric(lines[col], errors="coerce")

lines["line_total_usd"] = (
    lines["quantity"] * lines["list_price"] * (1 - lines["discount"])
)

lines[["order_id", "item_id", "quantity", "list_price", "discount", "line_total_usd"]].head()

,order_id,item_id,quantity,list_price,discount,line_total_usd
0,1,1,1,599.99,0.20,479.9920
1,1,2,2,1799.99,0.07,3347.9814
2,1,3,2,1549.00,0.05,2943.1000
3,1,4,2,599.99,0.05,1139.9810
4,1,5,1,2899.99,0.20,2319.9920


In [7]:
df.head()

,order_id,item_id,product_id,quantity,list_price,discount,extraction_timestamp,data_source,customer_id,order_status,...,data_source_order,line_total_usd,fx_snapshot_timestamp,fx_rate_USD,fx_rate_AED,line_total_aed,fx_rate_EUR,line_total_eur,fx_rate_GBP,line_total_gbp
0,1,1,20,1,599.99,0.20,2026-04-01 01:59:18.974759,PostgreSQl Server,259.0,4.0,...,PostgreSQl Server,479.9920,2026-04-01 21:42:12.252244,1.0,3.67315,1763.082615,0.863632,414.536451,0.751837,360.875745
1,1,2,8,2,1799.99,0.07,2026-04-01 01:59:18.974759,PostgreSQl Server,259.0,4.0,...,PostgreSQl Server,3347.9814,2026-04-01 21:42:12.252244,1.0,3.67315,12297.637879,0.863632,2891.423872,0.751837,2517.136292
2,1,3,10,2,1549.00,0.05,2026-04-01 01:59:18.974759,PostgreSQl Server,259.0,4.0,...,PostgreSQl Server,2943.1000,2026-04-01 21:42:12.252244,1.0,3.67315,10810.447765,0.863632,2541.755339,0.751837,2212.731475
3,1,4,16,2,599.99,0.05,2026-04-01 01:59:18.974759,PostgreSQl Server,259.0,4.0,...,PostgreSQl Server,1139.9810,2026-04-01 21:42:12.252244,1.0,3.67315,4187.321210,0.863632,984.524071,0.751837,857.079895
4,1,5,4,1,2899.99,0.20,2026-04-01 01:59:18.974759,PostgreSQl Server,259.0,4.0,...,PostgreSQl Server,2319.9920,2026-04-01 21:42:12.252244,1.0,3.67315,8521.678615,0.863632,2003.619331,0.751837,1744.255825
